In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "把输入翻译为{language}"),
    ("user", "{text}")
])

model = ChatOpenAI(
    model="doubao-seed-2.0-pro",
    base_url="https://ark.cn-beijing.volces.com/api/coding/v3")

parser = StrOutputParser()

# LCEL 表达式
# 构建链
chain = prompt_template | model | parser

print(chain.invoke({ "language": "中文", "text": "where are you" }))

analysis_prompt = ChatPromptTemplate.from_template("我应该怎么回复，{talk}。给一五个字的示例")

chain2 =  {"talk":chain} | analysis_prompt | model | parser

print(chain2.invoke({ "language": "中文", "text": "nice to meet you" }))


In [6]:
from langchain_core.runnables import RunnableMap, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


prompt_template_one = ChatPromptTemplate.from_messages([
    ("system", "把用户的输入翻译为英语"),
    ("user", "{text}")
])

prompt_template_two = ChatPromptTemplate.from_messages([
    ("system", "把用户的输入翻译为日语"),
    ("user", "{text}")
])

model = ChatOpenAI(
    model="doubao-seed-2.0-pro",
    base_url="https://ark.cn-beijing.volces.com/api/coding/v3")

parser = StrOutputParser()

# 构建链
chain_one = prompt_template_one | model | parser
chain_two = prompt_template_two | model | parser

# 并行执行
parallel_chains = RunnableMap({
    "one": chain_one,
    "two": chain_two,
})

# 合并结果
final_chain = parallel_chains | RunnableLambda(lambda x: f"英语：{x['one']}\n日语：{x['two']}")

final_chain.get_graph().print_ascii()

# 调用
# print(final_chain.invoke({"text":"你好"}))


              +------------------------+               
              | Parallel<one,two>Input |               
              +------------------------+               
                  ***             ***                  
                **                   **                
              **                       **              
+--------------------+          +--------------------+ 
| ChatPromptTemplate |          | ChatPromptTemplate | 
+--------------------+          +--------------------+ 
           *                               *           
           *                               *           
           *                               *           
    +------------+                  +------------+     
    | ChatOpenAI |                  | ChatOpenAI |     
    +------------+                  +------------+     
           *                               *           
           *                               *           
           *                               *    